# T33 — Is RSNA really an external test set? (overlap check)

The primary dataset's metadata says its whole Lung_Opacity class (6,012 images) and 8,851 of its Normal
images came from the RSNA pneumonia challenge — the same collection the "external" RSNA manifest is
built from. If those are the same images, an RSNA evaluation would mostly re-test training data and
cannot be reported as out-of-distribution robustness.

This notebook settles it with pixels, not counts. It makes a small fingerprint of every image
(64×64, brightness and contrast removed), finds for every primary image its most similar RSNA image,
and reports:
- how many RSNA-sourced primary images have a near-identical RSNA image;
- whether the matched RSNA image carries the matching label (Lung_Opacity ↔ "Lung Opacity",
  Normal ↔ "Normal"), which is an independent check that the matches are real;
- the same search for primary images that did **not** come from RSNA (the control group), to show
  what similarity unrelated chest X-rays reach;
- how many RSNA images the models saw in training, validation and test, and how many they never saw.

No model and no GPU are needed. **Expected run time:** about 15–25 minutes, almost all of it reading
the 26,684 RSNA DICOM files.

**Before running (Kaggle sidebar):**
1. Internet → **On**. An accelerator is not required.
2. Add input → the primary dataset `tawsifurrahman/covid19-radiography-database`.
3. Add input → Competitions → `rsna-pneumonia-detection-challenge` (you must have accepted its rules once).
4. Run All.

**Output:** `/kaggle/working/T33_rsna_overlap.zip` with `overlap_summary.json`, `rsna_primary_overlap.csv`
(one row per primary image: its best RSNA match and split), and two figures.

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path

REPO_URL    = "https://github.com/Ravindu-Pathirana/Chest-X-ray-Disease-Detection.git"
DATA_ROOT   = Path(os.environ.get("TRUST_DATA_ROOT", "/kaggle/input"))
WORK        = Path(os.environ.get("TRUST_WORK", "/kaggle/working"))
REPO        = Path(os.environ.get("TRUST_REPO", str(WORK / "Chest-X-ray-Disease-Detection")))
OUT         = WORK / "T33_rsna_overlap"
SIZE        = 64       # fingerprint side length
MATCH_SIM   = 0.97     # similarity above which two images are called the same. Unrelated chest X-rays reach about 0.94,
                       # an image against a resized copy of itself reaches 0.99+; the control group below checks this choice.
DRY_RUN     = os.environ.get("TRUST_DRY_RUN", "0") == "1"   # True: only 1,500 RSNA images, to try the notebook
OUT.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)], check=True)
for module, package in [("pydicom", "pydicom"), ("openpyxl", "openpyxl"), ("cv2", "opencv-python-headless")]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import pandas as pd
import cv2
import torch

In [ ]:
SKIP_DIRS = {"images", "masks", ".venv", ".git", "stage_2_train_images", "stage_2_test_images"}


def find_dir(root, test, what):
    for d, subdirs, _ in os.walk(root):
        p = Path(d)
        if test(p):
            return p
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS]
    raise FileNotFoundError(f"{what} not found under {root}")


PRIMARY = find_dir(DATA_ROOT, lambda p: (p / "COVID" / "images").is_dir() and (p / "Lung_Opacity" / "images").is_dir(),
                   "COVID-19_Radiography_Dataset (attach tawsifurrahman/covid19-radiography-database)")
RSNA = find_dir(DATA_ROOT, lambda p: (p / "stage_2_train_images").is_dir() and (p / "stage_2_detailed_class_info.csv").is_file(),
                "RSNA competition data (attach the competition rsna-pneumonia-detection-challenge)")
print("primary:", PRIMARY)
print("rsna:   ", RSNA)

# --- primary images with their recorded source and their split ------------------------------------
split = pd.read_csv(REPO / "artifacts/splits/split_manifest_v1.csv")
meta = []
for cls in ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]:
    m = pd.read_excel(PRIMARY / f"{cls}.metadata.xlsx")
    m["label"] = cls
    m["stem"] = m["FILE NAME"].astype(str).str.lower()
    meta.append(m[["stem", "label", "URL"]])
meta = pd.concat(meta)
split["stem"] = split.image_path.map(lambda p: Path(p).stem.lower())
primary = split.merge(meta, on=["stem", "label"], how="left", validate="one_to_one")
assert primary.URL.notna().all(), "some primary images have no metadata row"
primary["from_rsna"] = primary.URL.str.contains("rsna-pneumonia-detection-challenge")
print(pd.crosstab(primary.label, primary.from_rsna, margins=True))

# --- RSNA images with the challenge's own three-way class -------------------------------------------
rsna = pd.read_csv(RSNA / "stage_2_detailed_class_info.csv").drop_duplicates("patientId")
rsna = rsna.rename(columns={"patientId": "patient_id", "class": "rsna_class"})
manifest = pd.read_csv(REPO / "artifacts/segmentation/external_manifest_v1.csv")
rsna = rsna.merge(manifest[["patient_id", "label"]].rename(columns={"label": "manifest_label"}), on="patient_id", how="left")
print(pd.crosstab(rsna.rsna_class, rsna.manifest_label.fillna("not in manifest")))
if DRY_RUN:
    rsna = rsna.head(1500)
    primary = pd.concat([primary[primary.from_rsna].head(600), primary[~primary.from_rsna].head(300)])

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def fingerprint(img):
    """64x64, zero mean, unit length: the dot product of two fingerprints is their Pearson correlation,
    so a pure brightness or contrast change does not affect the match."""
    v = cv2.resize(img.astype(np.float32), (SIZE, SIZE), interpolation=cv2.INTER_AREA).ravel()
    v -= v.mean()
    return (v / (np.linalg.norm(v) + 1e-8)).astype(np.float32)


def primary_fp(rel_path):
    return fingerprint(cv2.imread(str(PRIMARY / rel_path), cv2.IMREAD_GRAYSCALE))


def rsna_fp(patient_id):
    import pydicom
    return fingerprint(pydicom.dcmread(RSNA / "stage_2_train_images" / f"{patient_id}.dcm").pixel_array)


def run_parallel(fn, items, label):
    t0, out = time.time(), []
    with ThreadPoolExecutor(max_workers=max(4, os.cpu_count() or 4)) as pool:
        for i, v in enumerate(pool.map(fn, items), 1):
            out.append(v)
            if i % 4000 == 0:
                print(f"  {label}: {i}/{len(items)}  ({time.time() - t0:.0f}s)")
    print(f"{label}: {len(items)} images in {time.time() - t0:.0f}s")
    return np.stack(out)


P = run_parallel(primary_fp, primary.image_path.tolist(), "primary")
R = run_parallel(rsna_fp, rsna.patient_id.tolist(), "rsna")
np.savez_compressed(OUT / "fingerprints.npz", primary=P.astype(np.float16), rsna=R.astype(np.float16))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Rt = torch.from_numpy(R).to(device)
best, second, arg = [], [], []
for i in range(0, len(P), 1024):
    sim = torch.from_numpy(P[i:i + 1024]).to(device) @ Rt.T
    top = sim.topk(2, dim=1)
    best.append(top.values[:, 0].cpu()); second.append(top.values[:, 1].cpu()); arg.append(top.indices[:, 0].cpu())
primary = primary.reset_index(drop=True)
primary["best_similarity"] = torch.cat(best).numpy()
primary["second_similarity"] = torch.cat(second).numpy()
idx = torch.cat(arg).numpy()
primary["rsna_patient_id"] = rsna.patient_id.to_numpy()[idx]
primary["rsna_class"] = rsna.rsna_class.to_numpy()[idx]
primary["matched"] = primary.best_similarity >= MATCH_SIM
expected = {"Lung_Opacity": "Lung Opacity", "Normal": "Normal"}
primary["label_agrees"] = [expected.get(l) == c for l, c in zip(primary.label, primary.rsna_class)]
primary.drop(columns=["stem"]).to_csv(OUT / "rsna_primary_overlap.csv", index=False)

control = primary[~primary.from_rsna]
src = primary[primary.from_rsna]
summary = {
    "match_threshold": MATCH_SIM,
    "control_images_not_from_rsna": int(len(control)),
    "control_best_similarity_median": float(control.best_similarity.median()),
    "control_best_similarity_max": float(control.best_similarity.max()),
    "control_matched": int(control.matched.sum()),
    "rsna_sourced_primary_images": int(len(src)),
    "rsna_sourced_matched": int(src.matched.sum()),
    "rsna_sourced_matched_fraction": float(src.matched.mean()),
    "rsna_sourced_best_similarity_median": float(src.best_similarity.median()),
    "label_agreement_among_matched": float(src[src.matched].label_agrees.mean()) if src.matched.any() else None,
    "distinct_rsna_images_matched": int(src[src.matched].rsna_patient_id.nunique()),
}
# Count every pixel match, whatever the metadata says about the image's source: on the first run the
# metadata turned out to attach the "not from RSNA" label to the wrong 1,341 Normal files.
all_matched = primary[primary.matched]
summary["primary_images_matched_to_rsna"] = int(len(all_matched))
summary["matched_but_metadata_says_not_rsna"] = int((all_matched.from_rsna == False).sum())
summary["label_agreement_among_all_matched"] = float(all_matched.label_agrees.mean()) if len(all_matched) else None
summary["max_similarity_of_unmatched"] = float(primary[~primary.matched].best_similarity.max())
seen = all_matched.drop_duplicates("rsna_patient_id")
by_split = seen.groupby(["label", "split"]).size().unstack(fill_value=0)
rsna["in_primary_split"] = rsna.patient_id.map(dict(zip(seen.rsna_patient_id, seen.split))).fillna("never seen")
exposure = pd.crosstab(rsna.rsna_class, rsna.in_primary_split, margins=True)
summary["rsna_images_total"] = int(len(rsna))
summary["rsna_images_in_primary_training_split"] = int((rsna.in_primary_split == "train").sum())
summary["rsna_images_never_seen"] = int((rsna.in_primary_split == "never seen").sum())
(OUT / "overlap_summary.json").write_text(json.dumps(summary, indent=2))
exposure.to_csv(OUT / "rsna_exposure_by_class.csv")

print(json.dumps(summary, indent=2))
print("\nMatched primary images by split:")
print(by_split)
print("\nRSNA images by the challenge's own class and where the models met them:")
print(exposure)
# Matches are trusted when they carry the matching RSNA label and are well separated from non-matches.
verdict = ("RSNA OVERLAPS the training data: it is not an external test set for these models."
           if len(all_matched) > 0.25 * len(primary) and summary["label_agreement_among_all_matched"] > 0.99
           and summary["max_similarity_of_unmatched"] < MATCH_SIM
           else "No clear overlap at this threshold: inspect the histogram and the example pairs before deciding.")
if summary["matched_but_metadata_says_not_rsna"]:
    print(f"NOTE: {summary['matched_but_metadata_says_not_rsna']} images the metadata lists as not from RSNA are "
          "pixel-identical to RSNA images. The metadata's per-file source is unreliable; use the pixel matches.")
print("\nVERDICT:", verdict)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.linspace(min(0.5, float(primary.best_similarity.min())), 1.0, 60)
ax.hist(control.best_similarity, bins=bins, alpha=0.7, label=f"control: primary images not from RSNA (n={len(control)})")
ax.hist(src.best_similarity, bins=bins, alpha=0.7, label=f"primary images recorded as from RSNA (n={len(src)})")
ax.axvline(MATCH_SIM, color="k", linestyle="--", linewidth=1)
ax.set_xlabel("similarity to the closest RSNA image"); ax.set_ylabel("images"); ax.set_yscale("log"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "similarity_histogram.png", dpi=120); plt.show()

import pydicom
pairs = pd.concat([src[src.matched].sample(min(4, int(src.matched.sum())), random_state=42),
                   control.sort_values("best_similarity", ascending=False).head(2)])
fig, axes = plt.subplots(2, len(pairs), figsize=(2.6 * len(pairs), 5.4), squeeze=False)
for c, r in enumerate(pairs.itertuples()):
    axes[0, c].imshow(cv2.imread(str(PRIMARY / r.image_path), cv2.IMREAD_GRAYSCALE), cmap="gray")
    axes[0, c].set_title(f"primary {r.label}\n{'from RSNA' if r.from_rsna else 'control'}", fontsize=8)
    axes[1, c].imshow(pydicom.dcmread(RSNA / "stage_2_train_images" / f"{r.rsna_patient_id}.dcm").pixel_array, cmap="gray")
    axes[1, c].set_title(f"closest RSNA: {r.rsna_class}\nsimilarity {r.best_similarity:.3f}", fontsize=8)
    axes[0, c].axis("off"); axes[1, c].axis("off")
plt.tight_layout(); plt.savefig(OUT / "example_pairs.png", dpi=120); plt.show()

In [ ]:
os.remove(OUT / "fingerprints.npz") if (OUT / "fingerprints.npz").stat().st_size > 400 * 2**20 else None
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Download this file from the Output panel:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MB)")